In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read input data from FRBNY website
us_gdp =  pd.read_excel("inputData/us_data.xlsx", sheet_name="gdp")
us_gdp = us_gdp.set_index("Date")
us_gdp['gdp.log'] = np.log(us_gdp['GDP'])
#drop gdp raw 
us_gdp = us_gdp.drop(columns = ['GDP'])
#convert index to datetime(e.g. 3/31/2025)
us_gdp.index = pd.to_datetime(us_gdp.index,)

us_gdp = us_gdp.resample("QS").mean()
us_gdp





,gdp.log
Date,
1947-01-01,7.688309
1947-04-01,7.685653
1947-07-01,7.683603
1947-10-01,7.699141
1948-01-01,7.714089
...,...
2023-10-01,10.041535
2024-01-01,10.045575
2024-04-01,10.052937


In [3]:
us_inflation =  pd.read_excel("inputData/us_data.xlsx", sheet_name='inflation')
us_inflation = us_inflation.set_index("Date")


#resample 'index' column to quarterly
us_inflation = us_inflation.resample('QS').mean()
us_inflation['growth'] = us_inflation.pct_change() * 100
pd.DataFrame(us_inflation)
#us_inflation = us_inflation.dropna()

us_inflation


# # #annualize quarterly growth
us_inflation['annualized'] =  100 * ((1 + us_inflation['growth']/100)**4 - 1)
# us_inflation['annualized']
us_inflation_final = us_inflation['annualized'].to_frame()
us_inflation_final['expectations'] = us_inflation_final.rolling(4).mean()
us_inflation_final = us_inflation_final.dropna()

us_inflation_final


,annualized,expectations
Date,,
1960-01-01,1.272464,2.111131
1960-04-01,1.523556,1.952745
1960-07-01,1.577147,1.671402
1960-10-01,1.317691,1.422715
1961-01-01,0.684357,1.275688
...,...,...
2024-01-01,3.749841,2.994231
2024-04-01,2.788488,2.729483
2024-07-01,2.190633,2.687933


In [4]:
#.interest    
us_interest =  pd.read_excel("inputData/us_data.xlsx", sheet_name = 'interest')
us_interest = us_interest.set_index("Date")
us_interest.index = pd.to_datetime(us_interest.index, format='%d/%m/%Y')
us_interest = us_interest.resample('QS').mean()
us_interest.columns = ['interest']
us_interest = us_interest.dropna()
us_interest

,interest
Date,
1954-07-01,1.500000
1954-10-01,1.500000
1955-01-01,1.500000
1955-04-01,1.710000
1955-07-01,1.966667
...,...
2024-01-01,5.330000
2024-04-01,5.330000
2024-07-01,5.263333


In [5]:
#merge us_gdp, us_inflation_final, us_interest
us_data = pd.concat([us_gdp, us_inflation_final, us_interest], axis=1).dropna()
us_data

,gdp.log,annualized,expectations,interest
Date,,,,
1960-01-01,8.165415,1.272464,2.111131,4.000000
1960-04-01,8.160017,1.523556,1.952745,3.883333
1960-07-01,8.164904,1.577147,1.671402,3.226667
1960-10-01,8.151990,1.317691,1.422715,3.000000
1961-01-01,8.158717,0.684357,1.275688,3.000000
...,...,...,...,...
2023-10-01,10.041535,2.022770,3.238626,5.330000
2024-01-01,10.045575,3.749841,2.994231,5.330000
2024-04-01,10.052937,2.788488,2.729483,5.330000


In [6]:
#rename braumns
us_data.columns = ['gdp.log', 'inflation', 'inflation.expectations', 'interest']

#add braumns of 0 called covid.indicator
us_data['covid.indicator'] = 0

#save data to excel
us_data.to_excel("inputData/Holston_Laubach_Williams_US.xlsx", sheet_name="US Input Data" )
